# Traffic Data Exploration — Los Angeles (2025)

Análise exploratória do dataset de tráfego processado (`traffic_2025_core.csv`).

**Fonte:** Caltrans PeMS — District 7 (Los Angeles)

**Objetivos:**
- Entender a estrutura e qualidade de cada coluna
- Verificar cobertura de dados por estação e rodovia
- Explorar padrões temporais de fluxo e velocidade (hora, dia, mês)
- Identificar rodovias mais congestionadas
- Entender a ausência de coordenadas e o que isso implica para o app

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

TRAFFIC_PATH = Path('..') / 'data' / 'processed' / 'traffic' / 'traffic_2025_core.csv'

# Arquivo tem 42M linhas — carrega em chunks
print("Carregando em chunks (pode levar ~30s)...")
chunks = []
for chunk in pd.read_csv(TRAFFIC_PATH, chunksize=500_000, low_memory=False):
    chunks.append(chunk)
df = pd.concat(chunks, ignore_index=True)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f'Registros  : {len(df):,}')
print(f'Período    : {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
print(f'Colunas    : {list(df.columns)}')

---
## 1. Estrutura e qualidade das colunas

In [ ]:
resumo = pd.DataFrame({
    'dtype'   : df.dtypes.astype(str),
    'nulos'   : df.isnull().sum(),
    'nulos_%' : (df.isnull().mean() * 100).round(1),
    'únicos'  : df.nunique(),
    'exemplo' : [str(df[c].dropna().iloc[0]) for c in df.columns],
})
print('=== RESUMO DAS COLUNAS ===')
print(resumo.to_string())

print('\n=== ESTATÍSTICAS NUMÉRICAS ===')
cols_num = ['station_length', 'percent_observed', 'total_flow', 'avg_occupancy', 'avg_speed']
print(df[cols_num].describe().round(3).to_string())

print(f'\n⚠️  Sem colunas lat/lon — localização via station_id + route + direction')

---
## 2. Estações e rodovias disponíveis

In [ ]:
print('=== ESTAÇÕES E RODOVIAS ===')
print(f'Estações únicas (station_id) : {df["station_id"].nunique():,}')
print(f'Rodovias (route)             : {df["route"].nunique()}  → {sorted(df["route"].unique().tolist())}')
print(f'Direções                     : {sorted(df["direction"].unique().tolist())}')

# Estações por rodovia
est_por_rota = df.groupby('route')['station_id'].nunique().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

est_por_rota.sort_values().plot(kind='barh', ax=axes[0], color='#1E88A8')
axes[0].set_title('Estações únicas por rodovia')
axes[0].set_xlabel('Nº de estações')
axes[0].set_ylabel('Route')

# Registros por direção
dir_counts = df.groupby('direction').size().sort_values(ascending=False)
dir_counts.plot(kind='bar', ax=axes[1], color='#F4821E')
axes[1].set_title('Registros por direção')
axes[1].set_xlabel('Direção'); axes[1].set_ylabel('Registros')
axes[1].tick_params(axis='x', rotation=0)
axes[1].bar_label(axes[1].containers[0], fmt='{:,.0f}', padding=3)

plt.tight_layout()
plt.show()

print('\nTop 10 rodovias por número de estações:')
print(est_por_rota.head(10).to_string())

---
## 3. Distribuições de fluxo, velocidade e ocupância

In [ ]:
df_v = df.dropna(subset=['total_flow', 'avg_speed']).copy()
print(f'Registros com total_flow + avg_speed válidos: {len(df_v):,}  ({len(df_v)/len(df)*100:.1f}% do total)')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# total_flow (clip p99 para não deixar outlier distorcer)
p99_flow = df_v['total_flow'].quantile(0.99)
axes[0].hist(df_v['total_flow'].clip(upper=p99_flow), bins=60,
             color='#1E88A8', edgecolor='white')
axes[0].set_title(f'Distribuição de total_flow\n(clip p99 = {p99_flow:,.0f} veíc/hora)')
axes[0].set_xlabel('Veículos/hora'); axes[0].set_ylabel('Contagem')

# avg_speed
axes[1].hist(df_v['avg_speed'], bins=60, color='#F4821E', edgecolor='white')
axes[1].axvline(df_v['avg_speed'].median(), color='red', linestyle='--',
                label=f'Mediana: {df_v["avg_speed"].median():.1f} mph')
axes[1].set_title('Distribuição de avg_speed')
axes[1].set_xlabel('Velocidade média (mph)'); axes[1].set_ylabel('Contagem')
axes[1].legend()

# avg_occupancy
axes[2].hist(df['avg_occupancy'].dropna(), bins=60, color='#4CAF50', edgecolor='white')
axes[2].set_title('Distribuição de avg_occupancy')
axes[2].set_xlabel('Ocupância (0–1)'); axes[2].set_ylabel('Contagem')

plt.suptitle('Distribuições das métricas de tráfego (PeMS 2025)', fontsize=13)
plt.tight_layout()
plt.show()

print('\n=== ESTATÍSTICAS (apenas registros com leitura válida) ===')
print(df_v[['total_flow', 'avg_speed', 'avg_occupancy']].describe().round(2).to_string())

---
## 4. Padrões temporais de fluxo e velocidade

In [ ]:
df['hour']        = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()
df['month']       = df['timestamp'].dt.month

ordem_dias  = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
nomes_dias  = ['Seg','Ter','Qua','Qui','Sex','Sáb','Dom']
nomes_meses = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

flow_hora  = df.groupby('hour')['total_flow'].mean()
speed_hora = df.groupby('hour')['avg_speed'].mean()
flow_dia   = df.groupby('day_of_week')['total_flow'].mean().reindex(ordem_dias)
speed_dia  = df.groupby('day_of_week')['avg_speed'].mean().reindex(ordem_dias)
flow_mes   = df.groupby('month')['total_flow'].mean()
speed_mes  = df.groupby('month')['avg_speed'].mean()

fig, axes = plt.subplots(3, 2, figsize=(16, 14))

# Fluxo por hora
axes[0,0].bar(flow_hora.index, flow_hora.values, color='#1E88A8')
axes[0,0].set_title('Fluxo médio por hora do dia')
axes[0,0].set_xlabel('Hora'); axes[0,0].set_ylabel('total_flow médio')
axes[0,0].set_xticks(range(0, 24))

# Velocidade por hora
axes[0,1].plot(speed_hora.index, speed_hora.values,
               color='#F4821E', linewidth=2, marker='o', markersize=4)
axes[0,1].set_title('Velocidade média por hora do dia')
axes[0,1].set_xlabel('Hora'); axes[0,1].set_ylabel('avg_speed (mph)')
axes[0,1].set_xticks(range(0, 24))

# Fluxo por dia da semana
axes[1,0].bar(range(7), flow_dia.values, color='#1E88A8')
axes[1,0].set_xticks(range(7)); axes[1,0].set_xticklabels(nomes_dias)
axes[1,0].set_title('Fluxo médio por dia da semana')
axes[1,0].set_ylabel('total_flow médio')

# Velocidade por dia da semana
axes[1,1].bar(range(7), speed_dia.values, color='#F4821E')
axes[1,1].set_xticks(range(7)); axes[1,1].set_xticklabels(nomes_dias)
axes[1,1].set_title('Velocidade média por dia da semana')
axes[1,1].set_ylabel('avg_speed (mph)')

# Fluxo por mês
axes[2,0].bar(range(12), flow_mes.values, color='#4CAF50')
axes[2,0].set_xticks(range(12)); axes[2,0].set_xticklabels(nomes_meses)
axes[2,0].set_title('Fluxo médio por mês')
axes[2,0].set_ylabel('total_flow médio')

# Velocidade por mês
axes[2,1].bar(range(12), speed_mes.values, color='#9C27B0')
axes[2,1].set_xticks(range(12)); axes[2,1].set_xticklabels(nomes_meses)
axes[2,1].set_title('Velocidade média por mês')
axes[2,1].set_ylabel('avg_speed (mph)')

plt.suptitle('Padrões temporais de tráfego — Los Angeles (2025)', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Hora de maior FLUXO     : {flow_hora.idxmax()}h  (flow médio = {flow_hora.max():,.0f})')
print(f'Hora de MENOR velocidade: {speed_hora.idxmin()}h  (speed = {speed_hora.min():.1f} mph)')
print(f'Hora de MAIOR velocidade: {speed_hora.idxmax()}h  (speed = {speed_hora.max():.1f} mph)')

In [ ]:
# Heatmap fluxo: hora × dia da semana
pivot_flow = df.pivot_table(
    values='total_flow', index='day_of_week', columns='hour', aggfunc='mean'
).reindex(ordem_dias)

# Heatmap velocidade: hora × dia da semana
pivot_speed = df.pivot_table(
    values='avg_speed', index='day_of_week', columns='hour', aggfunc='mean'
).reindex(ordem_dias)

fig, axes = plt.subplots(2, 1, figsize=(16, 9))

sns.heatmap(pivot_flow, cmap='Blues', ax=axes[0], linewidths=0.2,
            annot=False, cbar_kws={'label': 'Fluxo médio (veíc/hora)'})
axes[0].set_title('Heatmap — fluxo médio por hora × dia da semana')
axes[0].set_xlabel('Hora do dia')

sns.heatmap(pivot_speed, cmap='RdYlGn', ax=axes[1], linewidths=0.2,
            annot=False, cbar_kws={'label': 'Velocidade média (mph)'})
axes[1].set_title('Heatmap — velocidade média por hora × dia da semana\n(verde = livre, vermelho = congestionado)')
axes[1].set_xlabel('Hora do dia')

plt.tight_layout()
plt.show()

---
## 5. Rodovias mais congestionadas

In [ ]:
flow_rota  = df.groupby('route')['total_flow'].mean().sort_values(ascending=False)
speed_rota = df.groupby('route')['avg_speed'].mean().sort_values()
occ_rota   = df.groupby('route')['avg_occupancy'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

flow_rota.head(15).sort_values().plot(kind='barh', ax=axes[0], color='#1E88A8')
axes[0].set_title('Top 15 rodovias — maior FLUXO médio')
axes[0].set_xlabel('total_flow médio (veíc/hora)')
axes[0].set_ylabel('Route')

speed_rota.head(15).sort_values().plot(kind='barh', ax=axes[1], color='#E53935')
axes[1].set_title('Top 15 rodovias — MENOR velocidade\n(mais congestionadas)')
axes[1].set_xlabel('avg_speed (mph)')
axes[1].set_ylabel('Route')

occ_rota.head(15).sort_values().plot(kind='barh', ax=axes[2], color='#FF8F00')
axes[2].set_title('Top 15 rodovias — maior OCUPÂNCIA média')
axes[2].set_xlabel('avg_occupancy (0–1)')
axes[2].set_ylabel('Route')

plt.suptitle('Rodovias por nível de congestionamento (2025)', fontsize=13)
plt.tight_layout()
plt.show()

print('=== RANKING COMPLETO DE VELOCIDADE MÉDIA POR RODOVIA ===')
ranking = pd.DataFrame({
    'avg_speed' : df.groupby('route')['avg_speed'].mean().round(1),
    'total_flow': df.groupby('route')['total_flow'].mean().round(0),
    'avg_occ'   : df.groupby('route')['avg_occupancy'].mean().round(4),
}).sort_values('avg_speed')
print(ranking.to_string())

---
## 6. Cobertura dos sensores (percent_observed e nulos)

In [ ]:
print('=== NULOS POR COLUNA ===')
nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(1)
print(pd.DataFrame({'nulos': nulos, 'pct_%': pct}).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribuição de percent_observed
axes[0].hist(df['percent_observed'], bins=50, color='#1E88A8', edgecolor='white')
axes[0].set_title('Distribuição de percent_observed\n(% de intervalos com sensor ativo por hora)')
axes[0].set_xlabel('percent_observed (0–100)'); axes[0].set_ylabel('Contagem')

pct_zero = (df['percent_observed'] == 0).mean() * 100
pct_100  = (df['percent_observed'] == 100).mean() * 100
axes[0].text(0.97, 0.95, f'0% (sensor off): {pct_zero:.1f}%\n100% (completo): {pct_100:.1f}%',
             transform=axes[0].transAxes, ha='right', va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Cobertura de leitura por rodovia (% de linhas com total_flow não-nulo)
cobertura = df.groupby('route').apply(
    lambda g: g['total_flow'].notna().mean() * 100
).sort_values()
cobertura.plot(kind='barh', ax=axes[1], color='#4CAF50')
axes[1].set_title('Cobertura de leitura por rodovia\n(% de horas com total_flow não-nulo)')
axes[1].set_xlabel('% de horas com leitura')
axes[1].set_ylabel('Route')
axes[1].axvline(70, color='red', linestyle='--', label='70% threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nSensores com 0% de observação (apagados) : {(df["percent_observed"] == 0).sum():,}  ({pct_zero:.1f}%)')
print(f'Sensores com 100% de observação          : {(df["percent_observed"] == 100).sum():,}  ({pct_100:.1f}%)')

---
## 7. Conclusões para o app

### Estrutura do dataset (`traffic_2025_core.csv`)

| Coluna | Tipo | Cobertura | O que é |
|---|---|---|---|
| `timestamp` | datetime64 | 100% | Hora da medição (granularidade horária) |
| `station_id` | int64 | 100% | ID da estação PeMS (4.888 estações únicas) |
| `route` | int64 | 100% | Número da rodovia (23 freeways de LA) |
| `direction` | string | 100% | N / S / E / W |
| `station_length` | float | 57% | Comprimento do trecho monitorado (milhas) |
| `percent_observed` | int64 | 100% | % de intervalos de 5 min com sensor ativo |
| `total_flow` | float | 67% | Veículos contados na hora (NaN = sem leitura) |
| `avg_occupancy` | float | 67% | Ocupância da pista 0–1 |
| `avg_speed` | float | 57% | Velocidade média em mph |

### Principais achados

**Sem lat/lon — implicação crítica para o app:**
O dataset não tem coordenadas geográficas. As estações são identificadas apenas por `station_id + route + direction`. Para exibir tráfego no mapa do app é necessário baixar o arquivo de metadados do PeMS (`station_metadata.csv`) que mapeia cada `station_id` a uma localização geográfica.

**Nulos — o que significam:**
- `total_flow / avg_occupancy` (33% nulos): sensor não enviou leitura naquela hora — comum na rede PeMS. Não é dado corrompido.
- `avg_speed / station_length` (43% nulos): subconjunto menor de estações equipadas com sensor de velocidade.
- Filtrar por `percent_observed > 0` antes de qualquer análise para excluir sensores desligados.

**Padrão temporal:**
Dois picos de fluxo clássicos — **rush matutino (~8h)** e **rush vespertino (~17h)**. A velocidade cai nesses horários (congestionamento). Madrugada tem freeway livre (alta velocidade, baixo fluxo).

### Próximos passos para o app
1. Baixar metadados de estações PeMS para obter lat/lon de cada `station_id`
2. Fazer join com `traffic_2025_core.csv` para georreferenciar os segmentos
3. Usar `avg_speed` por `station_id` como input do heatmap de tráfego no Flutter
4. Filtrar apenas `percent_observed > 0` para garantir leituras válidas